# 04 — Paper 3 Figure 1 (study area)

Adapted from `Paper2_01_Fig1_Fig8_Maps.ipynb`. Run in Google Colab (Runtime ▸ Run all).

**What changed from Paper 2 Fig. 1**
- Adds the **111 tambon boundaries** (thin lines) under the 13 district boundaries (thick lines).
- Hatches the two tambons with **no HDC reporting facility** (Na Chong, Nong Saeng) — matches Sect. 2.1 / Table S1.
- Adds **all geocoded DIW-registered factories** (grey bubbles, sized by number of factories sharing a geocoded point) together with the **nine integrated cement plants** (numbered black squares).
- Adds a Thailand locator inset, north arrow and scale bar; width set to 174 mm (Springer double column).

**Inputs** (downloaded from Google Drive by file ID; sign in with the account that owns the files):
- `Saraburi_tambon_boundaries.geojson` — tambon polygons (same file used to match HDC tambon codes)
- `saraburi_factories_with_coords.csv` — DIW factory register with `lat`/`lon` (391 records, 384 geocoded in Paper 2)
- `osm_highway_1_2.geojson` — OSM Highway 1 and 2 centrelines (same as Papers 1–2)
- district boundaries and the Thailand outline from `prasertcbs/thailand_gis` (OCHA COD-AB, same as Papers 1–2)

Station coordinates: Air4Thai metadata (as in Papers 1–2). Cement plants: Global Energy Monitor (July 2026), as in Paper 2.

**Outputs:** `Figure_1.pdf`, `Figure_1.svg`, `Figure_1.png` (600 dpi), `Figure_1.tif` (600 dpi, LZW) — no in-figure title.

> **Repository note.** Files in this repository are numbered in run order (see the README). References to *Drive notebook NN* are to the author's original working notebooks, which are not part of the repository.

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available.
# Run locally: start Jupyter in the repository (or its notebooks/ folder).
# If data/processed/ is not found, the notebook falls back to the original Google Drive paths.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-healthcare-visits-saraburi"
USE_REPO = True          # False = use the original Google Drive paths only
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-healthcare-visits-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-healthcare-visits-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
REPO_DATA = os.path.join(REPO_DIR, "data", "processed") if REPO_DIR else ""
USE_REPO_DATA = bool(USE_REPO and REPO_DATA and os.path.isdir(REPO_DATA))

def repo_file(name):
    """Path of a file in data/processed/ (matched by file name), or None."""
    p = os.path.join(REPO_DATA, os.path.basename(str(name))) if USE_REPO_DATA else ""
    return p if p and os.path.exists(p) else None

print("Repository data:", REPO_DATA if USE_REPO_DATA else "not found - using Google Drive paths")

In [ ]:
!pip install -q geopandas gdown pyshp
!apt-get -qq install -y fonts-liberation > /dev/null

In [ ]:
import os, io, shutil, urllib.request
FILES = {'Saraburi_tambon_boundaries.geojson': '1KpaGEC-wor9kvTuFtinBRLm7Exvrtg5-',
         'saraburi_factories_with_coords.csv': '1v-aPM0QYmUEYOGIYeUkgL0WZJzijlnZ5',
         'osm_highway_1_2.geojson':            '1kpcZ33EdPeQnr0iydnzHjR6aBbzCiiQh'}
def fetch_from_drive(file_id, out):
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        auth.authenticate_user()
        req = build('drive', 'v3').files().get_media(fileId=file_id, supportsAllDrives=True)
        with io.FileIO(out, 'wb') as fh:
            dl, done = MediaIoBaseDownload(fh, req), False
            while not done: _, done = dl.next_chunk()
        print('Downloaded (Drive API):', out); return
    except Exception as ex:
        print('Drive API failed, trying gdown:', str(ex)[:100])
    import gdown; gdown.download(id=file_id, output=out, quiet=False)
for fname, fid in FILES.items():
    if os.path.exists(fname): continue
    if repo_file(fname): shutil.copy(repo_file(fname), fname); print('Repository data:', fname)
    else: fetch_from_drive(fid, fname)
URL = 'https://raw.githubusercontent.com/prasertcbs/thailand_gis/master/amphoe/thailand_province_amphoe_simplify.json'
if not os.path.exists('thailand_amphoe.json'): urllib.request.urlretrieve(URL, 'thailand_amphoe.json')
print(sorted(os.listdir('.')))

In [ ]:
# ---------- load and check layers ----------
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, geopandas as gpd

amp  = gpd.read_file('thailand_amphoe.json').set_crs(4326, allow_override=True)
prov = amp.dissolve(by='ADM1_EN').reset_index()            # all provinces (inset + neighbours)
roads = gpd.read_file('osm_highway_1_2.geojson').set_crs(4326, allow_override=True)
tam  = gpd.read_file('Saraburi_tambon_boundaries.geojson')
tam  = tam.set_crs(4326) if tam.crs is None else tam.to_crs(4326)
print('tambon columns:', tam.columns.tolist())
print('tambon polygons:', len(tam))

# pick the tambon-name and district-name columns automatically (print above to verify)
def pick(cols, keys):
    for k in keys:
        for c in cols:
            if c.lower() == k.lower(): return c
    for k in keys:
        for c in cols:
            if k.lower() in c.lower(): return c
    return None
cols = [c for c in tam.columns if c != 'geometry']
TAM_NAME = pick(cols, ['ADM3_EN', 'tambon_en', 'TAM_EN', 'NAME_3', 'tambon_e', 'T_NAME_E', 'tambon', 'ADM3_TH', 'tam_nam_t', 'T_NAME_T', 'name'])
AMP_NAME = pick(cols, ['ADM2_EN', 'amphoe_en', 'AMP_EN', 'NAME_2', 'amphoe_e', 'A_NAME_E', 'amphoe', 'ADM2_TH', 'amp_nam_t', 'A_NAME_T'])
print('tambon name column:', TAM_NAME, '| district column:', AMP_NAME)

# districts: dissolve the tambon file if it carries a district field (keeps lines coincident),
# otherwise fall back to the OCHA district layer
if AMP_NAME:
    dist = tam.dissolve(by=AMP_NAME).reset_index().rename(columns={AMP_NAME: 'DIST'})
    dist_src = 'dissolved from tambon file'
else:
    dist = amp[amp.ADM1_EN == 'Saraburi'].rename(columns={'ADM2_EN': 'DIST'})
    dist_src = 'OCHA COD-AB district layer'
print('districts:', len(dist), '(' + dist_src + ')')
sar_outline = tam.dissolve()

# the two tambons with no HDC reporting facility (Sect. 2.1, Table S1)
NO_HDC = {'na chong', 'nong saeng', 'หน้าช่อง', 'หนองแซง'}
def is_no_hdc(r):
    n = str(r[TAM_NAME]).strip().lower().replace('ต.', '').replace('ตำบล', '').strip()
    return n in NO_HDC
nohdc = tam[tam.apply(is_no_hdc, axis=1)] if TAM_NAME else tam.iloc[0:0]
print('no-HDC tambons found:', len(nohdc), nohdc[[TAM_NAME, AMP_NAME]].values.tolist() if AMP_NAME and len(nohdc) else '')
assert len(tam) == 111, 'Expected 111 tambons; check the boundary file.'
assert len(nohdc) == 2, 'Expected exactly 2 no-HDC tambons; set NO_HDC by code/name after checking the print-out.'

# DIW factories: keep points inside Saraburi; count factories per geocoded point
fac = pd.read_csv('saraburi_factories_with_coords.csv', encoding='utf-8-sig').dropna(subset=['lat', 'lon'])
fac_g = gpd.GeoDataFrame(fac, geometry=gpd.points_from_xy(fac.lon, fac.lat), crs=4326)
fac_in = gpd.sjoin(fac_g, sar_outline[['geometry']], predicate='within', how='inner')
cnt = fac_in.groupby(['lat', 'lon']).size().reset_index(name='n')
N_FAC, N_PTS = len(fac_in), len(cnt)
print(f'DIW factories: {len(fac)} geocoded, {N_FAC} inside Saraburi, at {N_PTS} distinct points (max {cnt.n.max()} per point)')

In [ ]:
# ---------- Figure 1 ----------
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, FancyArrow
import matplotlib.patheffects as pe
for f in fm.findSystemFonts():
    if 'LiberationSerif' in f: fm.fontManager.addfont(f)
plt.rcParams.update({'font.family': 'Liberation Serif', 'font.size': 8, 'svg.fonttype': 'none', 'pdf.fonttype': 42,
                     'hatch.linewidth': 0.5})
halo = [pe.withStroke(linewidth=2.2, foreground='white')]

STATIONS = pd.DataFrame({'name': ['24T', '25T'], 'lat': [14.685833, 14.526343], 'lon': [100.871996, 100.926047]})
PLANTS = pd.DataFrame({'no': range(1, 10),
    'name': ['TPI Polene Mittraphap', 'Siam Cement Kaeng Khoi', 'Khao Wong', 'Ta Luang', 'Siam City Thap Kwang 1',
             'Siam City Thap Kwang (8.45 Mt)', 'Siam City Thap Kwang (4.65 Mt)', 'Thai Pride Hin Son', 'Asia Cement Phu Khang'],
    'lat': [14.646310, 14.647975, 14.674128, 14.570239, 14.627492, 14.632372, 14.631969, 14.737328, 14.673757],
    'lon': [101.126496, 101.039987, 100.851875, 100.763862, 101.087613, 101.077219, 101.098706, 101.038138, 100.803369]})
RC = {'1': '#6A51A3', '2': '#D95F02'}
FAC_SCALE = 6      # marker area per factory sharing a point

x0, y0, x1, y1 = tam.total_bounds; pad = 0.04
EXT = (x0 - pad, x1 + pad, y0 - pad, y1 + pad)

fig, ax = plt.subplots(figsize=(6.85, 7.4))          # 174 mm wide
prov.plot(ax=ax, facecolor='#F2F2F2', edgecolor='#C8C8C8', linewidth=0.5, zorder=0)
tam.plot(ax=ax, facecolor='white', edgecolor='#BDBDBD', linewidth=0.3, zorder=0.5)
nohdc.plot(ax=ax, facecolor='none', edgecolor='#737373', hatch='////', linewidth=0.3, zorder=0.6)
dist.boundary.plot(ax=ax, edgecolor='#636363', linewidth=0.7, zorder=1)
sar_outline.boundary.plot(ax=ax, edgecolor='#252525', linewidth=1.2, zorder=1.1)
for _, r in dist.iterrows():
    pt = r.geometry.representative_point()
    ax.text(pt.x, pt.y, r.DIST, fontsize=6.5, color='#6B6B6B', style='italic', ha='center', va='center', zorder=1.5, path_effects=halo)
for _, r in roads.iterrows():
    gpd.GeoSeries([r.geometry]).plot(ax=ax, color=RC.get(str(r.ref), '#555555'), linewidth=1.5, zorder=3)
ax.scatter(cnt.lon, cnt.lat, s=cnt.n * FAC_SCALE, facecolor='#969696', edgecolor='#525252', linewidth=0.35, alpha=0.6, zorder=4)
ax.scatter(PLANTS.lon, PLANTS.lat, marker='s', s=30, facecolor='black', edgecolor='white', linewidth=0.7, zorder=6)
OFF = {1: (6, 4), 2: (-2, 7), 3: (-9, 6), 4: (-9, -9), 5: (2, -10), 6: (-9, -9), 7: (7, -6), 8: (5, 5), 9: (-10, 5)}
for _, r in PLANTS.iterrows():
    ax.annotate(str(r.no), (r.lon, r.lat), xytext=OFF[r.no], textcoords='offset points', fontsize=7,
                fontweight='bold', ha='center', va='center', zorder=7, path_effects=halo)
ax.scatter(STATIONS.lon, STATIONS.lat, marker='^', s=85, facecolor='#2171B5', edgecolor='black', linewidth=0.8, zorder=8)
for _, r in STATIONS.iterrows():
    ax.annotate(r['name'], (r.lon, r.lat), xytext=(8, -9), textcoords='offset points', fontsize=8.5, fontweight='bold',
                color='#08306B', zorder=9, path_effects=halo)
ax.set_xlim(EXT[0], EXT[1]); ax.set_ylim(EXT[2], EXT[3]); ax.set_aspect('equal')
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)'); ax.tick_params(labelsize=7.5)

# scale bar (20 km) and north arrow, lower-left / upper-right
lat_mid = (y0 + y1) / 2
deg_per_km = 1 / (111.32 * np.cos(np.radians(lat_mid)))
sx, sy = EXT[0] + 0.04, EXT[2] + 0.04
for i, c in enumerate(['black', 'white']):
    ax.add_patch(plt.Rectangle((sx + i * 10 * deg_per_km, sy), 10 * deg_per_km, 0.012, facecolor=c, edgecolor='black', lw=0.6, zorder=10))
for km in (0, 10, 20):
    ax.text(sx + km * deg_per_km, sy + 0.022, str(km), ha='center', va='bottom', fontsize=6.5, zorder=10)
ax.text(sx + 20 * deg_per_km + 0.015, sy + 0.006, 'km', ha='left', va='center', fontsize=6.5, zorder=10)
nx, ny = EXT[1] - 0.05, EXT[3] - 0.11
ax.add_patch(FancyArrow(nx, ny, 0, 0.05, width=0.012, head_width=0.03, head_length=0.025, color='black', zorder=10))
ax.text(nx, ny + 0.085, 'N', ha='center', va='bottom', fontsize=9, fontweight='bold', zorder=10)

# Thailand locator inset, upper-left
ins = ax.inset_axes([0.0, 0.70, 0.22, 0.30])
prov.plot(ax=ins, facecolor='#E0E0E0', edgecolor='#9E9E9E', linewidth=0.2)
prov[prov.ADM1_EN == 'Saraburi'].plot(ax=ins, facecolor='#D7301F', edgecolor='#D7301F', linewidth=0.3)
ins.set_xticks([]); ins.set_yticks([]); ins.set_xlabel(''); ins.set_ylabel(''); ins.set_aspect('equal'); ins.set_facecolor('white')
for s in ins.spines.values(): s.set_linewidth(0.6)
ins.text(0.05, 0.04, 'Thailand', transform=ins.transAxes, fontsize=6.5, style='italic')

# legend
k_ex = [k for k in (1, 5, 15) if k <= max(1, cnt.n.max())] or [1]
h = [Line2D([], [], marker='^', ls='', mfc='#2171B5', mec='black', ms=8, label='PCD monitoring station'),
     Line2D([], [], marker='s', ls='', mfc='black', mec='white', ms=6, label='Integrated cement plant (1–9)')]
h += [Line2D([], [], marker='o', ls='', mfc='#969696', mec='#525252', alpha=0.6, ms=np.sqrt(k * FAC_SCALE),
             label=f'{k} DIW-registered factor' + ('y' if k == 1 else 'ies') + ' per location') for k in k_ex]
h += [Line2D([], [], color=RC['1'], lw=1.5, label='Highway 1 (Phahonyothin Road)'),
      Line2D([], [], color=RC['2'], lw=1.5, label='Highway 2 (Mittraphap Road)'),
      Line2D([], [], color='#636363', lw=0.7, label='District boundary'),
      Line2D([], [], color='#BDBDBD', lw=0.6, label='Tambon boundary'),
      Patch(facecolor='white', edgecolor='#737373', hatch='////', lw=0.3, label='Tambon with no HDC reporting facility')]
fig.tight_layout()
leg = ax.legend(handles=h, loc='upper center', bbox_to_anchor=(0.5, -0.07), ncol=2, fontsize=7.5, frameon=False,
                handletextpad=0.5, columnspacing=1.2, labelspacing=0.7)

# plant key: placed under the legend's actual bottom edge (measured after drawing), split over 3 lines
fig.canvas.draw()
leg_bottom = leg.get_window_extent().transformed(fig.transFigure.inverted()).y0
rows = [PLANTS.iloc[i:i + 3] for i in range(0, 9, 3)]
KEY = '\n'.join('   '.join(f"{r['no']} {r['name']}" for _, r in g.iterrows()) for g in rows)
fig.text(0.5, leg_bottom - 0.012, 'Cement plants:\n' + KEY, ha='center', va='top', fontsize=6.8, linespacing=1.4)
for e in ('pdf', 'svg'): fig.savefig(f'Figure_1.{e}', bbox_inches='tight')
fig.savefig('Figure_1.png', dpi=600, bbox_inches='tight')
fig.savefig('Figure_1.tif', dpi=600, bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})
plt.close(fig)
print(f'saved Figure_1 (pdf, svg, png, tif) | DIW factories plotted: {N_FAC} at {N_PTS} locations')
from IPython.display import Image, display
display(Image('Figure_1.png', width=700))

In [ ]:
try:
    from google.colab import files
    for f in ['Figure_1.pdf', 'Figure_1.svg', 'Figure_1.png', 'Figure_1.tif']: files.download(f)
except ImportError:
    pass